### Data Sync

* **Dataset A** → raw truth, unformatted, growing, messy. New data appended here by import, manual or automation.

* **Dataset A_aligned** → latest duplicate of truth, intermediate layer, schema-aligned

* **Dataset B** → working copy → initial scenario for formatting, tests, validations and corrections.

* Sync rules must handle:

    - New rows in A that should appear in B.

    - Updated rows in A that should update B (without overwriting B’s corrections).

    - Corrections in B that eventually need to be reconciled back to A.

In [ ]:
from googleapiclient.errors import HttpError
import time

def sync_sheets(spreadsheet_id, aligned_sheet_name, db_test_sheet_name, synced_sheet_name):
    try:
        # Get sheet metadata to find sheet IDs
        spreadsheet = sheet_service.spreadsheets().get(
            spreadsheetId=spreadsheet_id,
            fields="sheets(properties(sheetId,title))"
        ).execute()
        
        # Find sheet IDs
        aligned_sheet_id = None
        for sheet in spreadsheet['sheets']:
            if sheet['properties']['title'] == aligned_sheet_name:
                aligned_sheet_id = sheet['properties']['sheetId']
                break
        
        if aligned_sheet_id is None:
            print(f"❌ Sheet '{aligned_sheet_name}' not found")
            return False
        
        # Get data from both sheets
        aligned_result = sheet_service.spreadsheets().values().get(
            spreadsheetId=spreadsheet_id,
            range=f"{aligned_sheet_name}"
        ).execute()
        
        db_test_result = sheet_service.spreadsheets().values().get(
            spreadsheetId=spreadsheet_id,
            range=f"{db_test_sheet_name}"
        ).execute()
        
        aligned_values = aligned_result.get('values', [])
        db_test_values = db_test_result.get('values', [])
        
        if not aligned_values:
            print("❌ No data found in aligned sheet")
            return False
        
        # Get headers (first row)
        headers = aligned_values[0]
        
        # Convert to dictionaries with uid as key
        aligned_dict = {}
        for row in aligned_values[1:]:
            if len(row) > 0:
                try:
                    uid_index = headers.index('uid')
                    uid = str(row[uid_index]) if uid_index < len(row) else ''
                    if uid:
                        aligned_dict[uid] = dict(zip(headers, row + [''] * (len(headers) - len(row))))
                except ValueError:
                    print("❌ 'uid' column not found in aligned sheet headers")
                    return False
        
        db_test_dict = {}
        for row in db_test_values[1:]:
            if len(row) > 0:
                try:
                    uid_index = headers.index('uid')
                    uid = str(row[uid_index]) if uid_index < len(row) else ''
                    if uid:
                        db_test_dict[uid] = dict(zip(headers, row + [''] * (len(headers) - len(row))))
                except ValueError:
                    print("❌ 'uid' column not found in db-test sheet headers")
                    return False
        
        # Merge data (db_test takes precedence)
        merged_dict = {**aligned_dict, **db_test_dict}
        
        # Convert back to list and sort by uid
        merged_data = list(merged_dict.values())
        merged_data.sort(key=lambda x: str(x.get('uid', '')))
        
        # Prepare values for writing
        values = [headers]
        for row in merged_data:
            values.append([row.get(header, '') for header in headers])
        
        # Check if synced sheet exists
        try:
            # Try to get the sheet to see if it exists
            sheet_service.spreadsheets().values().get(
                spreadsheetId=spreadsheet_id,
                range=f"{synced_sheet_name}!A1"
            ).execute()
            
            # Clear existing sheet if it exists
            clear_request = {
                "ranges": [f"{synced_sheet_name}"]
            }
            sheet_service.spreadsheets().values().batchClear(
                spreadsheetId=spreadsheet_id,
                body=clear_request
            ).execute()
            
        except HttpError:
            # Sheet doesn't exist, create it
            batch_update_request = {
                'requests': [{
                    'addSheet': {
                        'properties': {
                            'title': synced_sheet_name
                        }
                    }
                }]
            }
            sheet_service.spreadsheets().batchUpdate(
                spreadsheetId=spreadsheet_id,
                body=batch_update_request
            ).execute()
            print(f"✅ Created new sheet: {synced_sheet_name}")
        
        # Write data to synced sheet
        update_request = {
            'values': values
        }
        sheet_service.spreadsheets().values().update(
            spreadsheetId=spreadsheet_id,
            range=f"{synced_sheet_name}",
            valueInputOption='RAW',
            body=update_request
        ).execute()
        
        # Copy formatting from aligned sheet
        copy_format_requests = [
            {
                'copyPaste': {
                    'source': {
                        'sheetId': aligned_sheet_id,
                        'startRowIndex': 0,
                        'endRowIndex': 1,  # Header row
                        'startColumnIndex': 0,
                        'endColumnIndex': len(headers)
                    },
                    'destination': {
                        'sheetId': get_sheet_id(spreadsheet_id, synced_sheet_name),
                        'startRowIndex': 0,
                        'endRowIndex': 1,
                        'startColumnIndex': 0,
                        'endColumnIndex': len(headers)
                    },
                    'pasteType': 'PASTE_FORMAT'
                }
            },
            {
                'copyPaste': {
                    'source': {
                        'sheetId': aligned_sheet_id,
                        'startRowIndex': 1,  # Data rows
                        'endRowIndex': len(aligned_values),
                        'startColumnIndex': 0,
                        'endColumnIndex': len(headers)
                    },
                    'destination': {
                        'sheetId': get_sheet_id(spreadsheet_id, synced_sheet_name),
                        'startRowIndex': 1,
                        'endRowIndex': len(values),
                        'startColumnIndex': 0,
                        'endColumnIndex': len(headers)
                    },
                    'pasteType': 'PASTE_FORMAT'
                }
            }
        ]
        
        # Apply formatting
        batch_update_request = {
            'requests': copy_format_requests
        }
        
        sheet_service.spreadsheets().batchUpdate(
            spreadsheetId=spreadsheet_id,
            body=batch_update_request
        ).execute()
        
        # Auto-resize columns to fit content
        auto_resize_request = {
            'requests': [{
                'autoResizeDimensions': {
                    'dimensions': {
                        'sheetId': get_sheet_id(spreadsheet_id, synced_sheet_name),
                        'dimension': 'COLUMNS',
                        'startIndex': 0,
                        'endIndex': len(headers)
                    }
                }
            }]
        }
        
        sheet_service.spreadsheets().batchUpdate(
            spreadsheetId=spreadsheet_id,
            body=auto_resize_request
        ).execute()
        
        print(f"✅ Successfully created/updated synced sheet '{synced_sheet_name}' with formatting")
        print(f"📊 Total records: {len(merged_data)}")
        print(f"📁 Records from db-test: {len(db_test_dict)}")
        print(f"📁 Additional records from aligned: {len(merged_data) - len(db_test_dict)}")
        
        return True
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        return False

def get_sheet_id(spreadsheet_id, sheet_name):
    """Helper function to get sheet ID by name"""
    spreadsheet = sheet_service.spreadsheets().get(
        spreadsheetId=spreadsheet_id,
        fields="sheets(properties(sheetId,title))"
    ).execute()
    
    for sheet in spreadsheet['sheets']:
        if sheet['properties']['title'] == sheet_name:
            return sheet['properties']['sheetId']
    
    raise ValueError(f"Sheet '{sheet_name}' not found")

# Call the function
sync_sheets(
    spreadsheet_id=ORIGINAL_SPREADSHEET_ID,
    aligned_sheet_name="aligned",
    db_test_sheet_name="db-test",
    synced_sheet_name="synced"
)